# 01 - Explore Iceberg Reference Data

Browse the 4 Iceberg tables that power the rightsizing engine:
- `rs_cloud_sku_catalog` - Instance types, specs, pricing
- `rs_cloud_sku_pricing_history` - CUR-derived effective prices
- `rs_finops_service_config` - 1,048+ service configs
- `rs_rightsizing_rules` - Sizing rules (19 generic + 26 Azure)

Also explores Bronze/Silver layer tables for debugging pipeline data.

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from local_helpers import athena_query, LocalIcebergReader
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

reader = LocalIcebergReader()
CLOUD = 'aws'  # Change to 'azure', 'gcp', 'oci', 'alibaba' as needed

## 1. SKU Catalog

In [2]:
catalog_df = reader.load_sku_catalog(CLOUD)
print(f'Shape: {catalog_df.shape}')
print(f'Columns: {list(catalog_df.columns)}')
catalog_df.head(10)

Shape: (0, 29)
Columns: ['cloud_provider', 'service_category', 'service_name', 'sku_code', 'sku_family', 'sku_generation', 'sku_size_ordinal', 'tier', 'region', 'vcpus', 'memory_gb', 'gpu_count', 'gpu_type', 'storage_gb', 'storage_type', 'max_iops', 'max_throughput_mbps', 'network_bandwidth_gbps', 'hourly_price', 'monthly_price', 'pricing_unit', 'pricing_model', 'currency', 'specs', 'is_active', 'is_burstable', 'last_updated', 'source', 'job_runtime_utc']


,cloud_provider,service_category,service_name,sku_code,sku_family,sku_generation,sku_size_ordinal,tier,region,vcpus,memory_gb,gpu_count,gpu_type,storage_gb,storage_type,max_iops,max_throughput_mbps,network_bandwidth_gbps,hourly_price,monthly_price,pricing_unit,pricing_model,currency,specs,is_active,is_burstable,last_updated,source,job_runtime_utc


In [3]:
# Distribution by service_name
catalog_df['service_name'].value_counts().head(20)

Series([], Name: count, dtype: int64)

In [4]:
# Look up a specific SKU
sku = 'm5.xlarge'
catalog_df[catalog_df['sku_code'] == sku]

,cloud_provider,service_category,service_name,sku_code,sku_family,sku_generation,sku_size_ordinal,tier,region,vcpus,memory_gb,gpu_count,gpu_type,storage_gb,storage_type,max_iops,max_throughput_mbps,network_bandwidth_gbps,hourly_price,monthly_price,pricing_unit,pricing_model,currency,specs,is_active,is_burstable,last_updated,source,job_runtime_utc


## 2. Pricing History (CUR-derived)

In [5]:
pricing_df = reader.load_pricing_history(CLOUD, lookback_days=30)
print(f'Shape: {pricing_df.shape}')
print(f'Columns: {list(pricing_df.columns)}')
pricing_df.head(10)

Shape: (0, 16)
Columns: ['cloud_provider', 'service_name', 'sku_code', 'region', 'pricing_model', 'effective_price', 'list_price', 'pricing_unit', 'currency', 'account_id', 'usage_quantity', 'usage_date', 'record_count', 'last_updated', 'year_month', 'job_runtime_utc']


,cloud_provider,service_name,sku_code,region,pricing_model,effective_price,list_price,pricing_unit,currency,account_id,usage_quantity,usage_date,record_count,last_updated,year_month,job_runtime_utc


In [6]:
# Price trend for a specific SKU
sku = 'm5.xlarge'
sku_prices = pricing_df[pricing_df['sku_code'] == sku].sort_values('usage_date')
if not sku_prices.empty:
    print(f'Price history for {sku}:')
    display(sku_prices[['sku_code', 'region', 'effective_hourly_price', 'usage_date']].tail(10))
else:
    print(f'No pricing data for {sku}')

No pricing data for m5.xlarge


## 3. Service Configs

In [7]:
configs_df = reader.load_service_configs(CLOUD)
print(f'Shape: {configs_df.shape}')
print(f'Columns: {list(configs_df.columns)}')
configs_df.head(20)

Shape: (13, 18)
Columns: ['cloud_provider', 'service_category', 'service_name', 'primary_metric', 'unit_of_measurement', 'underutilization_indicator', 'recommended_actions', 'threshold_source', 'additional_notes', 'threshold_config', 'sizing_strategy', 'sizing_dimension', 'cur_service_code', 'cur_sku_field', 'is_active', 'created_at', 'updated_at', 'job_runtime_utc']


,cloud_provider,service_category,service_name,primary_metric,unit_of_measurement,underutilization_indicator,recommended_actions,threshold_source,additional_notes,threshold_config,sizing_strategy,sizing_dimension,cur_service_code,cur_sku_field,is_active,created_at,updated_at,job_runtime_utc
0,aws,AI/ML,Amazon SageMaker,CPU Utilization,percent,CPU utilization < 10% over 7 days,Downsize instance type or terminate idle endpoints,programmatic_defaults,NaN,"{""conditions"": [{""metric"": ""cpu"", ""operator"": ""lt"", ""threshold"": 10, ""unit"":...",downsize,vcpu,AmazonSageMaker,line_item_usage_type,True,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00
1,aws,Analytics,Amazon OpenSearch Service,CPU Utilization,percent,CPU utilization < 15% over 14 days,Downsize instance type or reduce data nodes,programmatic_defaults,NaN,"{""conditions"": [{""metric"": ""cpu"", ""operator"": ""lt"", ""threshold"": 15, ""unit"":...",downsize,vcpu,AmazonES,line_item_usage_type,True,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00
2,aws,Analytics,Amazon Redshift,CPU Utilization,percent,CPU utilization < 15% over 7 days,Downsize cluster or pause when idle,programmatic_defaults,NaN,"{""conditions"": [{""metric"": ""cpu"", ""operator"": ""lt"", ""threshold"": 15, ""unit"":...",downsize,vcpu,AmazonRedshift,line_item_usage_type,True,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00
3,aws,Cache,Amazon ElastiCache,CPU Utilization,percent,CPU utilization < 15% over 14 days,Downsize node type or reduce cluster size,programmatic_defaults,NaN,"{""conditions"": [{""metric"": ""cpu"", ""operator"": ""lt"", ""threshold"": 15, ""unit"":...",downsize,vcpu,AmazonElastiCache,line_item_usage_type,True,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00
4,aws,Compute,Amazon EC2,CPU Utilization,percent,CPU utilization < 10% over 14 days,Downsize instance type or terminate idle instances,programmatic_defaults,NaN,"{""conditions"": [{""metric"": ""cpu"", ""operator"": ""lt"", ""threshold"": 10, ""unit"":...",downsize,vcpu,AmazonEC2,line_item_usage_type,True,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00
5,aws,Containers,Amazon ECS,CPU Utilization,percent,CPU utilization < 20% over 14 days,Downsize task definitions or use Fargate Spot,programmatic_defaults,NaN,"{""conditions"": [{""metric"": ""cpu"", ""operator"": ""lt"", ""threshold"": 20, ""unit"":...",downsize,vcpu,AmazonECS,line_item_usage_type,True,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00
6,aws,Containers,Amazon EKS,CPU Utilization,percent,CPU utilization < 20% over 14 days,Downsize node groups or use Fargate for low-use workloads,programmatic_defaults,NaN,"{""conditions"": [{""metric"": ""cpu"", ""operator"": ""lt"", ""threshold"": 20, ""unit"":...",downsize,vcpu,AmazonEKS,line_item_usage_type,True,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00
7,aws,Databases,Amazon DynamoDB,Read/Write Capacity,RCU/WCU,Provisioned capacity utilization < 20% over 7 days,Switch to on-demand or reduce provisioned capacity,programmatic_defaults,NaN,"{""conditions"": [{""metric"": ""request_units"", ""operator"": ""lt"", ""threshold"": 2...",downsize,capacity,AmazonDynamoDB,line_item_usage_type,True,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00
8,aws,Databases,Amazon RDS,CPU Utilization,percent,CPU utilization < 15% over 14 days,Downsize instance class or use Aurora Serverless,programmatic_defaults,NaN,"{""conditions"": [{""metric"": ""cpu"", ""operator"": ""lt"", ""threshold"": 15, ""unit"":...",downsize,vcpu,AmazonRDS,line_item_usage_type,True,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+00:00,2026-02-27 22:51:26.938649+

In [8]:
# Service categories breakdown
configs_df['service_category'].value_counts()

service_category
Analytics     2
Storage       2
Containers    2
Databases     2
AI/ML         1
Compute       1
Cache         1
Networking    1
Serverless    1
Name: count, dtype: int64

## 4. Rightsizing Rules

In [9]:
rules_df = reader.load_rules(CLOUD)
print(f'Shape: {rules_df.shape}')
print(f'Columns: {list(rules_df.columns)}')
rules_df

Shape: (19, 11)
Columns: ['cloud_provider', 'service_category', 'service_name', 'rule_code', 'action_type', 'sizing_method', 'sizing_params', 'default_savings_pct', 'priority', 'is_active', 'job_runtime_utc']


,cloud_provider,service_category,service_name,rule_code,action_type,sizing_method,sizing_params,default_savings_pct,priority,is_active,job_runtime_utc
0,aws,Storage,NaN,STORAGE_IDLE_TERMINATE,terminate,terminate,{},100.0,1,True,2026-02-27 22:51:23.818095+00:00
1,aws,Compute,NaN,COMPUTE_IDLE_TERMINATE,terminate,terminate,{},100.0,1,True,2026-02-27 22:51:23.818095+00:00
2,aws,AI/ML,NaN,ML_IDLE_TERMINATE,terminate,terminate,{},100.0,1,True,2026-02-27 22:51:23.818095+00:00
3,aws,Databases,NaN,DB_IDLE_TERMINATE,terminate,terminate,{},100.0,1,True,2026-02-27 22:51:23.818095+00:00
4,aws,Networking,NaN,NET_IDLE_TERMINATE,terminate,terminate,{},100.0,1,True,2026-02-27 22:51:23.818095+00:00
5,aws,Networking,NaN,NET_TIER_DOWNGRADE,downsize_tier,tier_downgrade,{},25.0,2,True,2026-02-27 22:51:23.818095+00:00
6,aws,Compute,NaN,COMPUTE_RIGHTSIZE_STEP_DOWN,downsize_vcpu,step_down,"{""steps"": 1, ""min_vcpu"": 1, ""min_memory_gb"": 0.5}",30.0,2,True,2026-02-27 22:51:23.818095+00:00
7,aws,Containers,NaN,CONTAINER_RIGHTSIZE,downsize_vcpu,step_down,"{""steps"": 1, ""min_vcpu"": 1, ""min_memory_gb"": 0.5}",25.0,2,True,2026-02-27 22:51:23.818095+00:00
8,aws,Security,NaN,SEC_TIER_DOWNGRADE,downsize_tier,tier_downgrade,{},20.0,2,True,2026-02-27 22:51:23.818095+00:00
9,aws,AI/ML,NaN,ML_RIGHTSIZE,downsize_vcpu,step_down,"{""steps"": 1, ""min_vcpu"": 2, ""min_memory_gb"": 4.0}",30.0,2,True,2026-02-27 22:51:23.818095+00:00


In [10]:
# Rules by sizing method
if not rules_df.empty:
    rules_df.groupby(['service_category', 'sizing_method']).size().reset_index(name='count')

## 5. Browse Bronze / Silver Tables

In [11]:
# List all Iceberg tables
db = os.getenv('ICEBERG_DB', 'finomics_catalog_data')
all_tables = athena_query(f'SHOW TABLES IN {db}')
all_tables

,tab_name
0,bronze_alibaba_tbl
1,bronze_aws_cloud_recomm_tbl
2,bronze_aws_cur_report_tbl
3,bronze_aws_focus_report_tbl
4,bronze_aws_service_component_tbl
...,...
163,silver_oci_account_service_summary_tbl
164,silver_oci_cost_usage_merged_tbl
165,silver_oci_recommendations_summary_tbl
166,silver_oci_region_cost_analysis_tbl


In [12]:
# Sample from bronze EC2 instances (change table name as needed)
athena_query("""
    SELECT * FROM bronze_aws_ec2_instances
    LIMIT 5
""")

OperationalError: TABLE_NOT_FOUND: line 1:15: Table 'awsdatacatalog.prod_finomics_catalog_data.bronze_aws_ec2_instances' does not exist

In [ ]:
# Sample from bronze CloudWatch metrics
athena_query("""
    SELECT * FROM bronze_aws_cloudwatch_metrics
    LIMIT 5
""")

In [ ]:
# Sample from silver recommendations
athena_query("""
    SELECT * FROM silver_aws_custom_recommendations
    LIMIT 10
""")

## 6. Ad-hoc Athena Queries

Use `athena_query()` for any custom SQL against Iceberg tables.

In [ ]:
# Your ad-hoc query here
athena_query("""
    SELECT cloud_provider, COUNT(*) as total_skus,
           COUNT(DISTINCT service_name) as services
    FROM rs_cloud_sku_catalog
    WHERE is_active = TRUE
    GROUP BY cloud_provider
    ORDER BY total_skus DESC
""")